### Two-deme Wright-Fisher forward-in-time simulation with migration
#### 1.1 Autosomes
This is a two-deme, single-locus, two-allele forward simulator. There are several input arguments: <code>Nm</code> and <code>Nf</code>, the male and female populations sizes of the two demes. <code>Nm=c(100, 200)</code> means there are 100 and 200 males in the first and second deme respectively. Note that are vectors of length two. 

For sex-specific migration rates, we have <code>Mm</code> and <code>Mf</code> in the form of 2x2 matrices, with rows sum to one. For instance, 
$$
M_m=\begin{bmatrix}
0.95 & 0.05\\
0.1 & 0.9
\end{bmatrix}
$$
means 5% of gametes in the first deme comes from the parents of the second deme, and 10% of the gametes in the second deme are from the opposite deme. 

<code>t</code> is the number of generations to be simulated forward in time. All initial allele frequencies are set to 0.5. Allele frequency does not matter as long as the locus remains highly polymorphic. 

Outputs are the allele frequency vectors over time for the two demes, separated by sex. Note that this is a population-based (instead of individual-based) simulator thus only allele freuqencies are recorded. We are missing individual genotypic information, but in return the simulator is 10x faster. We do not consider the effect of sampling for objective #1. 

In [2]:
# Nm: MALE SIZES IN POP1 AND POP2 (A VECTOR)
# Nf: FEMALE SIZES IN POP1 AND POP2 (ANOTHER VECTOR)
# Mm: MIGRATION MATRIX FOR MALES (2X2 MATRIX)
# Mf: MIGRATION MATRIX FOR FEMALES (2X2 MATRIX)
# t: NUMBER OF GENERATIONS TO SIMULATE FORWARD IN TIME, SCALER
sim_autosomes<-function(Nm=c(100, 100), Nf=c(100, 100), 
            Mm=matrix(c(0.95, 0.05, 0.05, 0.95), nc=2), Mf=matrix(c(0.95, 0.05, 0.05, 0.95), nc=2), 
            t=10)
{
    # INITIALISATION
    # EMPTY VECTORS TO BE FILLED IN. pop1m = ALLELE FREQ OF MALES IN POP1
    pop1m<-rep(NA, t+1)
    pop1f<-rep(NA, t+1)
    pop2m<-rep(NA, t+1)
    pop2f<-rep(NA, t+1)
    # INITIAL FREQ, ALL AT 0.5
    pop1m[1]<-0.5; pop1f[1]<-0.5; pop2m[1]<-0.5; pop2f[1]<-0.5;
    # PROPAGATION
    for (i in 1:t)
    {
        # NEW GAMETE POOL FREQ FOR THE DIFFERENT COMPARTMENTS, AFTER MIGRATION
        gametem<-Mm%*%c(pop1m[i], pop2m[i])
        gametef<-Mf%*%c(pop1f[i], pop2f[i])
        # SAMPLING OFFSPRING. OFFSPRING COMES FROM ONE PAIR OF MALE AND FEMALE PARENTS
        pop1m[i+1]<-(rbinom(1, size=Nm[1], prob=gametem[1])+rbinom(1, size=Nm[1], prob=gametef[1]))/(2*Nm[1])
        pop1f[i+1]<-(rbinom(1, size=Nf[1], prob=gametem[1])+rbinom(1, size=Nf[1], prob=gametef[1]))/(2*Nf[1])
        pop2m[i+1]<-(rbinom(1, size=Nm[2], prob=gametem[2])+rbinom(1, size=Nm[2], prob=gametef[2]))/(2*Nm[2])
        pop2f[i+1]<-(rbinom(1, size=Nf[2], prob=gametem[2])+rbinom(1, size=Nf[2], prob=gametef[2]))/(2*Nf[2])
    }
    # RETURN ALL THE ALLELE FREQ
    return(list(pop1m=pop1m, pop1f=pop1f, pop2m=pop2m, pop2f=pop2f))
}

In [3]:
# TEST RUN
sim_autosomes()

$pop1m
 [1] 0.500 0.550 0.475 0.475 0.445 0.435 0.400 0.540 0.485 0.470 0.495

$pop1f
 [1] 0.500 0.480 0.500 0.485 0.470 0.465 0.485 0.415 0.485 0.465 0.460

$pop2m
 [1] 0.500 0.550 0.540 0.500 0.495 0.590 0.510 0.505 0.450 0.425 0.380

$pop2f
 [1] 0.500 0.540 0.550 0.500 0.570 0.525 0.515 0.520 0.580 0.530 0.440

#### 1.2 Chromosome X
One key difference between chromosome X and autosomes is their numbers in males and females: Males have XY, females have XX. When sampling a new male offspring, the X must be inherited from the maternal side (and Y from paternal, although not simulated). 

Extension: in a multilocus system, loci on XY are unable to recombine thus the effective recombination rate on X will be lower. 

In [4]:
# Nm: MALE SIZES IN POP1 AND POP2 (A VECTOR)
# Nf: FEMALE SIZES IN POP1 AND POP2 (ANOTHER VECTOR)
# Mm: MIGRATION MATRIX FOR MALES (2X2 MATRIX)
# Mf: MIGRATION MATRIX FOR FEMALES (2X2 MATRIX)
# t: NUMBER OF GENERATIONS TO SIMULATE FORWARD IN TIME
sim_X<-function(Nm=c(100, 100), Nf=c(100, 100), 
            Mm=matrix(c(0.95, 0.05, 0.05, 0.95), nc=2), Mf=matrix(c(0.95, 0.05, 0.05, 0.95), nc=2), 
            t=10)
{
    # INITIALISATION. pop1m = ALLELE FREQ OF MALES IN POP1
    pop1m<-rep(NA, t+1)
    pop1f<-rep(NA, t+1)
    pop2m<-rep(NA, t+1)
    pop2f<-rep(NA, t+1)
    pop1m[1]<-0.5; pop1f[1]<-0.5; pop2m[1]<-0.5; pop2f[1]<-0.5;
    # PROPAGATION
    for (i in 1:t)
    {
        # NEW GAMETE POOL FREQ, AFTER CONSIDERING SEX-SPECIFIC MIGRATION RATES
        gametem<-Mm%*%c(pop1m[i], pop2m[i])
        gametef<-Mf%*%c(pop1f[i], pop2f[i])
        # SAMPLING OFFSPRING
        pop1m[i+1]<-rbinom(1, size=Nm[1], prob=gametef[1])/Nm[1]
        pop1f[i+1]<-(rbinom(1, size=Nf[1], prob=gametem[1])+rbinom(1, size=Nf[1], prob=gametef[1]))/(2*Nf[1])
        pop2m[i+1]<-rbinom(1, size=Nm[2], prob=gametef[2])/Nm[2]
        pop2f[i+1]<-(rbinom(1, size=Nf[2], prob=gametem[2])+rbinom(1, size=Nf[2], prob=gametef[2]))/(2*Nf[2])
    }
    return(list(pop1m=pop1m, pop1f=pop1f, pop2m=pop2m, pop2f=pop2f))
}

In [5]:
# TEST RUN
sim_X()

$pop1m
 [1] 0.50 0.64 0.51 0.53 0.66 0.53 0.59 0.60 0.58 0.55 0.51

$pop1f
 [1] 0.500 0.530 0.535 0.550 0.560 0.580 0.570 0.565 0.535 0.570 0.525

$pop2m
 [1] 0.50 0.52 0.50 0.45 0.45 0.56 0.38 0.50 0.40 0.37 0.39

$pop2f
 [1] 0.500 0.475 0.505 0.480 0.495 0.420 0.500 0.380 0.425 0.400 0.430

### 2. Build-up of FST over time
With initial frequencies of 0.5 the model assumes the two demes are one unified population at the beginning. They split into two but maintain some gene flow, specified by the two migration matrices. There are two forces, drift and migration, acting on the system. Drift is private to each deme thus pushes them away, while migration (i.e. the sharing of genetic contents) pulls them back as one population. 

FST is a measure of population differentiation. Without considering the effect of sampling its simplest form is: 
$$F_{ST}=\frac{(p_1-p_2)^2}{p(1-p)}$$

$F_{ST}=0$ means no segregation (panmixia?); $F_{ST}=1$ means they are in complete isolation. There should be an equilibrium over time. 

Due to the stochastic nature the main spirit is to run multiple repeats (for the same set of parameters) and then look at the average allele frequencies (or FST or any other summary statistics) per time point. 